# MSIT — Xây dựng dữ liệu SFT (AGTD / CTTD / Test) trên Kaggle

Notebook này dựng lại quy trình sinh dữ liệu của MSIT nhưng **không định nghĩa lại logic**:
toàn bộ code nằm trong package `msit`, notebook chỉ cấu hình đường dẫn Kaggle và gọi hàm.

| Bước | Module được gọi |
| --- | --- |
| Đọc annotation OAMine | `msit.data.readers.OAMineRawReader` |
| Lọc nhiễu, cân bằng danh mục, chia tập động | `msit.data.preparator.MSITDataPreparator` |
| Sinh nhãn bằng Qwen2.5-VL 4-bit (local) | `msit.models.qwen_local.QwenLocalDataGenerator` |
| Ghi JSONL có thể chạy tiếp (resume) | `msit.data.sft_labels` |
| Driver đầu-cuối | `msit.data.build_sft` (`python -m msit.data.build_sft`) |

**Thiết lập Kaggle:**
1. Settings → Accelerator: `GPU T4 x2` hoặc `GPU P100` (cần ≥ 16 GB VRAM cho Qwen2.5-VL-7B 4-bit).
2. Settings → Internet: `ON` (tải model từ HuggingFace).
3. Add-ons → Datasets: gắn dataset chứa `datasets/oamine/annotations/*.jsonl` và `datasets/images/oamine/<asin>.jpg`.

File đầu ra ghi vào `/kaggle/working/` để không mất khi session kết thúc:
`splits_checkpoint.json`, `test_set.jsonl`, `agtd_train.jsonl`, `cttd_train.jsonl`.

In [ ]:
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
    )
# T4/P100 không hỗ trợ bf16 -> QwenLocalDataGenerator nạp fp16 + 4-bit NF4

In [ ]:
# Cài thư viện và lấy source package. Bỏ qua bước clone nếu repo đã có sẵn.
!pip install -q accelerate bitsandbytes transformers qwen-vl-utils pillow torchvision

import os

REPO_DIR = os.environ.get("MSIT_REPO_DIR", "/kaggle/working/msit-open-world")
if not os.path.isdir(os.path.join(REPO_DIR, "msit")):
    !git clone https://github.com/linhnnh688/msit-open-world.git {REPO_DIR}

# Khi chạy local (Jupyter trong repo), tự dò thư mục gốc chứa package msit/
if not os.path.isdir(os.path.join(REPO_DIR, "msit")):
    _cand = os.path.abspath(os.getcwd())
    while True:
        if os.path.isfile(os.path.join(_cand, "msit", "__init__.py")):
            REPO_DIR = _cand
            break
        _parent = os.path.dirname(_cand)
        if _parent == _cand:
            break
        _cand = _parent

import sys

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print("REPO_DIR =", REPO_DIR)

In [ ]:
# Đường dẫn I/O: mặc định theo layout Kaggle, có thể override bằng biến môi trường.
import os

from msit.data.build_sft import DEFAULT_INPUT_DIR, DEFAULT_WORKING_DIR, SFTBuildConfig

INPUT_DIR = os.environ.get("MSIT_INPUT_DIR", DEFAULT_INPUT_DIR)
WORKING_DIR = os.environ.get("MSIT_WORKING_DIR", DEFAULT_WORKING_DIR)

cfg = SFTBuildConfig.from_dirs(INPUT_DIR, WORKING_DIR, seed=42)
print("OAMine annotations :", cfg.oamine_dir)
print("Ảnh sản phẩm       :", cfg.image_dir)
print("Checkpoint         :", cfg.checkpoint_file)
print("Test set           :", cfg.test_set_file)
print("AGTD / CTTD out    :", cfg.agtd_out_file, "/", cfg.cttd_out_file)

In [ ]:
# Đọc annotation + chia tập. Lần chạy đầu tạo splits_checkpoint.json;
# các lần sau (session bị restart) đọc lại checkpoint, không chia lại.
from msit.data.build_sft import load_or_create_splits

test_pool, agtd_pool, cttd_pool = load_or_create_splits(cfg, rebuild=False)

print(f"Test: {len(test_pool)} | AGTD: {len(agtd_pool)} | CTTD: {len(cttd_pool)}")
print(
    "Ví dụ record:",
    {k: v for k, v in (agtd_pool or test_pool or [{}])[0].items() if k != "raw"},
)

In [ ]:
# Ghi test set (ghi đè mỗi lần chạy).
from msit.data.sft_labels import save_test_set

save_test_set(test_pool, cfg.test_set_file)

In [ ]:
# Nạp Qwen2.5-VL-7B-Instruct 4-bit cho phần sinh nhãn.
from msit.models.qwen_local import DEFAULT_MODEL_ID, QwenLocalDataGenerator

MODEL_ID = os.environ.get("MSIT_MODEL_ID", DEFAULT_MODEL_ID)
generator = QwenLocalDataGenerator(model_id=MODEL_ID)

In [ ]:
# Sinh nhãn AGTD (trích thuộc tính text + ảnh rồi merge) và CTTD (CoT 5 bước).
# Hàm ghi nối tiếp theo ASIN: chạy lại notebook sẽ bỏ qua mẫu đã sinh xong.
from msit.data.sft_labels import generate_and_save_sft_labels

n_agtd = generate_and_save_sft_labels(
    agtd_pool, cfg.agtd_out_file, mode="agtd", generator=generator
)

n_cttd = generate_and_save_sft_labels(
    cttd_pool, cfg.cttd_out_file, mode="cttd", generator=generator
)

print(f"\nMẫu mới sinh trong lần chạy này: AGTD={n_agtd}, CTTD={n_cttd}")
print(
    "Huấn luyện dữ liệu Local GPU hoàn tất. Hãy tải các file jsonl từ mục Output về máy."
)

### Kiểm tra kết quả

```python
!ls -lh /kaggle/working/*.jsonl /kaggle/working/splits_checkpoint.json
!wc -l /kaggle/working/test_set.jsonl /kaggle/working/agtd_train.jsonl /kaggle/working/cttd_train.jsonl
```

Ghi chú:
- `attribute_to_judge = "target_attribute_placeholder"` ở nửa số mẫu CTTD "yes" vẫn là TODA trong
  `msit/data/preparator.py` (cần thay bằng key thuộc tính thật bóc từ dữ liệu gốc).
- Mẫu CTTD "no" dùng thuộc tính giả từ danh mục khác (contrastive sampling, paper Section 3.2).
- Muốn chạy toàn bộ pipeline bằng một lệnh: `python -m msit.data.build_sft --input-dir ... --working-dir ...`.